In [7]:
# ml-service-python/data/generate_synthetic.py
import json
import random
from pathlib import Path

# OUT = Path(__file__).parent / "kb_texts.jsonl"
OUT = Path.cwd() / "kb_texts.jsonl"

diagnoses = [
    "Normal cognition",
    "Very Mild Alzheimer's Disease",
    "Mild Alzheimer's Disease",
    "Moderate Alzheimer's Disease",
    "Severe Alzheimer's Disease"
]

apoe_choices = ["e3/e3", "e3/e4", "e4/e4", "e2/e3"]
sex_choices = ["M", "F"]

def gen_case(i):
    age = random.randint(55, 90)
    mmse = max(0, min(30, int(random.gauss(24 - (age-70)/6, 3))))
    cdr = round(min(3, max(0, random.choice([0,0.5,1,2,3]) if mmse < 26 else 0)),1)
    etiv = int(random.gauss(1500, 80))
    nwbv = round(max(0.5, min(0.9, random.gauss(0.70 - (age-65)/300, 0.04))), 3)
    apoe = random.choice(apoe_choices)
    sex = random.choice(sex_choices)
    # map mmse -> coarse diagnosis
    if mmse >= 27:
        diag = "Normal cognition"
    elif mmse >= 24:
        diag = "Very Mild Alzheimer's Disease"
    elif mmse >= 20:
        diag = "Mild Alzheimer's Disease"
    elif mmse >= 14:
        diag = "Moderate Alzheimer's Disease"
    else:
        diag = "Severe Alzheimer's Disease"

    text = (
        f"CaseID: {i}. Age: {age} years, Sex: {sex}, MMSE: {mmse}, CDR: {cdr}, "
        f"eTIV: {etiv} mL, nWBV: {nwbv}. APOE: {apoe}. Outcome: {diag}."
    )
    return {"case_id": f"case_{i:05d}", "text": text, "age": age, "mmse": mmse, "cdr": cdr, "diag": diag}

def main(n=2000):
    OUT.parent.mkdir(parents=True, exist_ok=True)
    with OUT.open("w", encoding="utf8") as f:
        for i in range(1, n+1):
            case = gen_case(i)
            f.write(json.dumps(case, ensure_ascii=False) + "\n")
    print(f"Wrote {n} cases to {OUT}")

if __name__ == "__main__":
    main(2000)


Wrote 2000 cases to /content/kb_texts.jsonl


In [ ]:
from google.colab import files
files.download(str(OUT))